In [ ]:
!pip install python-docx -q
print('python-docx installed')

In [ ]:
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT

doc = Document()
doc.styles['Normal'].font.name = 'Calibri'
doc.styles['Normal'].font.size = Pt(11)

# Title page
p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = p.add_run('INTERVIEW MASTER GUIDE')
r.bold = True
r.font.size = Pt(26)
doc.add_paragraph('')
p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = p.add_run('production-agentic-rag')
r.bold = True
r.font.size = Pt(18)
doc.add_paragraph('')
p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
p.add_run('Deep technical interview preparation for the actual project implementation')
doc.add_page_break()

# TOC-style overview
doc.add_heading('TABLE OF CONTENTS', level=1)
toc = [
    '1. Project Overview',
    '2. Architecture',
    '3. Complete Request Lifecycle',
    '4. File-by-File Codebase Explanation',
    '5. Agentic RAG vs Basic RAG',
    '6. Planner',
    '7. Retrieval',
    '8. Guardrails',
    '9. Tools',
    '10. Graph / Orchestration',
    '11. LLM + Embedding Providers',
    '12. Ingestion',
    '13. Evaluation',
    '14. Observability',
    '15. Testing',
    '16. API',
    '17. Design Decisions',
    '18. Trade-Offs',
    '19. Failure Modes',
    '20. Security',
    '21. Productionization',
    '22. Interview Questions',
    '23. Interviewer Attack Mode',
    '24. Trick Questions',
    '25. Key Concepts I Must Know',
    '26. What I Should Say About the Project',
    '27. Project Walkthrough Script',
    '28. 1-Day Interview Preparation',
    '29. Final Cheat Sheet'
]
for item in toc:
    doc.add_paragraph(item)
doc.add_page_break()

def section(title):
    doc.add_heading(title, level=1)

def add_bullets(items):
    for item in items:
        doc.add_paragraph(item, style='List Bullet')

def add_note(text):
    p = doc.add_paragraph()
    p.add_run('NOTE: ').bold = True
    p.add_run(text)

def add_table(rows, header=None):
    table = doc.add_table(rows=len(rows), cols=len(rows[0]))
    for r_idx, row in enumerate(rows):
        for c_idx, val in enumerate(row):
            cell = table.cell(r_idx, c_idx)
            cell.text = str(val)
            if r_idx == 0 and header is not False:
                for p in cell.paragraphs:
                    for run in p.runs:
                        run.bold = True
    return table

# Section 1
section('1. PROJECT OVERVIEW')
doc.add_paragraph('Project name: production-agentic-rag')
doc.add_paragraph('Problem being solved: this project addresses the gap between toy RAG demos and production-grade knowledge systems. Instead of just retrieving top-k chunks and dumping them into a prompt, it adds planning, multi-hop retrieval, grounding checks, evaluation, and guardrails so a question-answering system can behave more like a controlled agent.)
doc.add_paragraph('Why the project exists: the README states that most RAG demos stop at “embed → top-k → prompt,” while production systems need planning, multi-hop retrieval, grounding, validation, observability, and evaluation. This repo turns those concerns into explicit modules, with offline deterministic defaults so the system can be tested in CI without API keys.')
doc.add_paragraph('Target users and use case: the package is designed for an internal or domain-specific knowledge assistant over a corpus like product docs, handbook content, policy information, support knowledge, or short document sets. The sample corpus in data/corpus/handbook.md and the eval dataset in data/eval/qa.jsonl show a support/operations use case.')
doc.add_paragraph('Core capabilities: hybrid retrieval, multi-hop query planning, self-correction, grounded synthesis, provider abstraction, guardrails, evaluation harness, FastAPI service, LangGraph optional engine, and observability/tracing.')
doc.add_paragraph('What makes it different from ordinary RAG: the retrieval is not only lexical+dense fusion; the system explicitly decomposes queries, collects citations, tracks faithfulness, and retries with reformulated queries when grounding is weak. The project is not just retrieval + prompting; it is a small agentic workflow with guardrails and a measurement layer.')
doc.add_paragraph('Technology stack: Python 3.10+, pure-Python core, optional OpenAI/Azure OpenAI, FastAPI/Uvicorn, LangGraph, pytest, Docker, and GitHub Actions. The core runs without runtime dependencies because the default LLM is a deterministic mock and embeddings use hashing.')
doc.add_paragraph('Architecture style: modular monolith / orchestration pipeline. There is a top-level RAGPipeline that assembles and calls providers, retriever, and agent. The agent logic is explicit and the LangGraph engine is an optional, alternate execution path. The design emphasizes pluggability and controlled evaluation.')
doc.add_paragraph('Major components: providers, ingestion, retrieval, agents, guardrails, observability, evaluation, API, graph, and tests.')
doc.add_paragraph('30-second interview pitch: “This project is an agentic RAG system that ingests a document corpus, plans multi-hop retrieval across sub-questions, fuses BM25 and dense ranking, validates grounding, and synthesizes an answer with citations. It is designed to be production-aware rather than a toy demo: it includes guardrails, observability, provider abstraction, and an evaluation harness, with a default offline mock mode so the whole system can be tested without external APIs.”')
doc.add_paragraph('2-minute interview explanation: “The project is a knowledge assistant built around a RAGPipeline. Documents are loaded and chunked, then indexed with hybrid retrieval using BM25 plus vector similarity and reciprocal-rank fusion. A planner decomposes the user question into sub-questions, and an AgenticRAG loop executes retrieval per hop, deduplicates citations, calls the LLM to synthesize grounded output, and measures faithfulness. If grounding is weak, the agent reformulates the query and retries. The system also blocks prompt-injection inputs, redacts PII in answers, and produces trace summaries with token and cost accounting. The core is intentionally provider-agnostic so it can run with a mock LLM or switch to OpenAI/Azure. The evaluation harness runs labeled QA cases and computes retrieval metrics plus faithfulness and answer correctness.”')
doc.add_paragraph('5–10 minute technical explanation: “At the project root, RAGPipeline coordinates the pipeline. Settings are loaded from environment variables via Settings, and the pipeline builds a HybridRetriever with a provider-specific embedder and an LLM provider. It then wraps a RetrievalTool around the retriever and creates an AgenticRAG instance. During query execution, check_input rejects suspicious prompts and blocked inputs. The agent calls plan(question), which splits the question into sub-questions using a heuristic regex-based decomposition. For each sub-question it invokes the retrieval tool, deduplicates the retrieved chunks, and aggregates all retrieved chunk texts. The LLM is asked to answer using only those texts. The project computes grounding_score by comparing the answer tokens to the retrieved context tokens, and if the score falls below a faithfulness threshold, it self-corrects by appending the retrieved context to the original query and retrying. The optional LangGraph engine implements the same flow as a StateGraph with nodes for plan, retrieve, generate, reformulate, and conditional edges. The project is deliberately not a basic RAG demo because it treats retrieval and generation as an iterative, stateful reasoning loop with observability and metric collection.”')

# Section 2
section('2. ARCHITECTURE')
doc.add_paragraph('High-level system architecture:')
diagram = '''
User
  |
  v
API / CLI / SDK
  |
  v
RAGPipeline
  |
  +--> check_input -> InputBlocked guardrail
  |
  +--> AgenticRAG / LangGraphRAG
         |
         +--> plan()
         +--> RetrievalTool -> HybridRetriever
                +--> BM25
                +--> VectorStore
                +--> RRF fusion + rerank
         +--> LLMProvider.complete()
         +--> grounding_score
         +--> reformulate retry
  |
  v
answer + citations + trace
'''
p = doc.add_paragraph()
p.add_run(diagram)
p.runs[0].font.name = 'Consolas'
p.runs[0].font.size = Pt(10)
doc.add_paragraph('Each component and why it exists: the ingestion layer converts documents to chunks, the retrieval layer finds relevant evidence, the agent reasons over sub-questions, the LLM synthesizes a grounded answer, and the observation/evaluation layer tracks latency, token cost, and result quality. The project keeps each concern isolated so it can be swapped or tested independently.')
doc.add_paragraph('Complete end-to-end request flow: query enters check_input, builds a Tracer, calls pipeline.query, runs engine.run, collects contexts, computes faithfulness, redacts PII, and returns QueryResult with answer, citations, contexts, faithfulness, iterations, trace.')
doc.add_paragraph('Agentic RAG flow: plan -> retrieve -> synthesize -> score -> retry if needed. This is the core differentiator from simple one-shot RAG.')
doc.add_paragraph('Ingestion pipeline: loaders -> chunking -> index. Retrieval pipeline: lexical + dense -> RRF -> rerank -> candidate set. Planner -> Agent -> Tools flow: plan() decomposes, tool is a search_knowledge_base function, agent loops. Guardrail flow: check_input before processing, grounding_score after generation, redact_pii on final answer. Evaluation architecture: run_eval over labelled QA dataset; metrics computed on citations and answer correctness. Observability flow: tracer spans record latency contributions and usage cost. Failure/recovery flow: input block; empty retrieval yields low grounding; faithfulness gate triggers reformulation; exceptions fall back gracefully if LangGraph absent.')

# Section 3
section('3. COMPLETE REQUEST LIFECYCLE')
doc.add_paragraph('Example: a complex user query like “What is the return window and how long do refunds take?” enters the pipeline.')
steps = [
    'User asks a question.',
    'API / CLI / direct call invokes RAGPipeline.query(question).',
    'RAGPipeline creates a Tracer and calls check_input(question).',
    'If blocked, query returns QueryResult(blocked=True) and stops.',
    'If allowed, AgenticRAG.run(question, tracer) starts.',
    'plan(query) splits the request into sub-questions; in the project this is a regex-based heuristic, not an LLM planner.',
    'For each sub-question, RetrievalTool calls HybridRetriever.retrieve().',
    'HybridRetriever does BM25 search and VectorStore cosine search, then reciprocal rank fusion on both result sets.',
    'Optional rerank adds overlap and phrase bonuses, then results are trimmed to top-k.',
    'The agent deduplicates retrieved chunks by chunk.id, builds a context list, and calls llm.complete(system_prompt, prompt).',
    'The answer is scored using grounding_score(answer, contexts).',
    'If faithfulness is below threshold, the agent calls _reformulate(question, contexts) and runs another iteration up to max_iterations.',
    'Final best answer is kept; citations come from seen chunk ids.',
    'The pipeline redacts PII and computes final faithfulness, then returns QueryResult with citations, answer, contexts, and trace summary.'
]
add_bullets(steps)
doc.add_paragraph('Files and functions involved: in src/production_agentic_rag/pipeline.py, RAGPipeline.query calls self._engine.run; in agents/rag_agent.py, AgenticRAG.run performs the loop; in agents/planner.py plan decomposes the request; in retrieval/hybrid.py HybridRetriever.retrieve handles ranking; in guardrails.py grounding_score and redact_pii do validation; in observability.py Tracer records spans and token cost.')

# Section 4
section('4. FILE-BY-FILE CODEBASE EXPLANATION')
file_map = [
    ['api/main.py', 'FastAPI service; exposes GET /health and POST /ask; builds a shared RAGPipeline and loads the corpus once.'],
    ['src/production_agentic_rag/config.py', 'Settings dataclass loads environment variables for engines, providers, chunk sizes, top_k, rerank, and faithfulness threshold.'],
    ['src/production_agentic_rag/pipeline.py', 'Top-level orchestrator. Initializes retriever, LLM, tool, agent, and optional LangGraph engine, applies input guardrails, and emits QueryResult.'],
    ['src/production_agentic_rag/graph.py', 'Optional LangGraph StateGraph engine with plan -> retrieve -> generate -> reformulate loop and conditional routing.'],
    ['src/production_agentic_rag/guardrails.py', 'Input prompt-injection detection, length limits, PII redaction, and grounding_score.'],
    ['src/production_agentic_rag/observability.py', 'Tracer spans, token usage accounting, and cost estimation.'],
    ['src/production_agentic_rag/agents/planner.py', 'Heuristic sub-question decomposition using regex splits.'],
    ['src/production_agentic_rag/agents/rag_agent.py', 'AgenticRAG loop: plan, retrieve, synthesize, score, reformulate.'],
    ['src/production_agentic_rag/agents/tools.py', 'RetrievalTool wrappers retrieval as a callable tool.'],
    ['src/production_agentic_rag/providers/llm.py', 'MockLLM and OpenAILLM behind the LLMProvider protocol.'],
    ['src/production_agentic_rag/providers/embeddings.py', 'HashingEmbedder and OpenAIEmbedder behind the Embedder protocol.'],
    ['src/production_agentic_rag/retrieval/bm25.py', 'BM25 implementation from scratch with IDF and length normalization.'],
    ['src/production_agentic_rag/retrieval/vectorstore.py', 'Dense vector store with cosine similarity and embedding cache.'],
    ['src/production_agentic_rag/retrieval/hybrid.py', 'HybridRetriever uses RRF and reranking.'],
    ['src/production_agentic_rag/ingestion/loaders.py', 'Document and load_directory helpers for .md and .txt files.'],
    ['src/production_agentic_rag/ingestion/chunking.py', 'Chunk dataclass plus fixed-size and markdown-section chunkers.'],
    ['src/production_agentic_rag/text.py', 'Tokenization, stemming, stopword filtering, and cosine similarity helpers.'],
    ['src/production_agentic_rag/evaluation/metrics.py', 'Retrieval metrics and faithfulness/answer_similarity formulas.'],
    ['src/production_agentic_rag/evaluation/harness.py', 'Dataset loader and run_eval pipeline.'],
    ['src/production_agentic_rag/integrations.py', 'Optional gateway and observability adapters with graceful fallback.'],
    ['data/eval/qa.jsonl', 'Sample QA evaluation dataset with relevant_ids.'],
    ['tests/test_agent.py', 'Tests planner, citations, and grounded multi-hop behavior.'],
    ['tests/test_guardrails.py', 'Tests blocking injection, PII redaction, and grounding estimates.'],
    ['tests/test_pipeline.py', 'Tests end-to-end query and evaluation harness quality.'],
    ['tests/test_retrieval.py', 'Tests chunk overlap, BM25 ranking, and hybrid retrieval.'],
    ['tests/test_langgraph.py', 'Tests optional LangGraph path and default built-in engine fallback.'],
    ['tests/test_integrations.py', 'Tests observability export fallback and optional gateway integration.'],
    ['tests/test_text.py', 'Tests tokenization/stemming and cosine behavior.']
]
add_table(file_map, header=False)
doc.add_paragraph('The key point is that the project is intentionally structured around interfaces: LLMProvider, Embedder, Chunker, retrieval tool, and engine selections. The code favors explicit wiring and observable behavior over hidden magical abstractions.')

# Section 5
section('5. AGENTIC RAG VS BASIC RAG')
rows = [
    ['Dimension', 'Basic RAG', 'This project'],
    ['Query handling', 'Single retrieval call or one-shot prompt', 'Query is decomposed into sub-questions and optionally retried'],
    ['Retrieval', 'One top-k retrieval pass', 'BM25 + vector retrieval + reciprocal rank fusion + rerank'],
    ['Generation', 'Prompt with retrieved docs', 'Prompt with retrieved docs, but also groundedness scoring'],
    ['Planning', 'Not explicit', 'plan() is an explicit reasoning step'],
    ['Tool use', 'Usually none', 'Retrieval is exposed as a callable tool'],
    ['State', 'Low or no state', 'Agent maintains query, contexts, iterations, citations'],
    ['Self-correction', 'Usually absent', 'reformulate + faithfulness gate if score is low'],
    ['Guardrails', 'Rare', 'Input length/injection checks, PII redaction, grounding checks'],
    ['Evaluation', 'Often ad hoc', 'Dataset harness with retrieval and answer metrics'],
    ['Observability', 'Often absent', 'Tracer spans, usage accounting, costs']
]
add_table(rows)
doc.add_paragraph('Implementation reality: the repository is agentic in a constrained, practical sense. It does not have a full autonomous tool ecosystem or arbitrary external action execution; it has structured multi-hop retrieval, query reformulation, and a tool-wrapped retrieval capability. That is enough to qualify as agentic RAG under a pragmatic definition, but it is not a fully autonomous general-purpose agent.')
doc.add_paragraph('IMPLEMENTED: planner decomposition, tool calling, iterative self-correction, stateful loops, provider abstraction, guardrails, tracing, evaluation harness.
PARTIALLY IMPLEMENTED: dynamic tool selection beyond retrieval is limited; the planner is heuristic, not LLM-based.
NOT IMPLEMENTED: arbitrary permissioned tool graph, session memory, human-in-the-loop approval, or persistent stateful agents.
RECOMMENDED EXTENSION: add real LLM planner, cross-encoder reranker, authenticated tool gateway, response caching, and safer multi-tenant deployment boundaries.')

# Section 6
section('6. PLANNER')
doc.add_paragraph('The planner exists to break a complex query into smaller retrieval steps, reducing the chance that a single broad question overwhelms the retrieval system. In this project, planner.py defines plan(question), which regex-splits on question marks, conjunctions, semicolons, and the word “then”. It returns a list of sub-questions and falls back to the original question if nothing meaningful is found.')
doc.add_paragraph('Why it exists: the repo is designed for multi-hop, grounded retrieval. This is especially useful when a query contains multiple facts, such as “return window and refunds take” or “shipping and warranty.” It prevents a single large query from flattening everything into one fuzzy retrieval request.')
doc.add_paragraph('What it receives: the raw question string. What it produces: a list[str] of sub-questions. How it decides: regex-based decomposition; deterministic and simple. Is it LLM-based or hybrid? The current implementation is deterministic heuristic, but the README notes that the interface is LLM-swappable. The planner contract is very easy to replace.')
doc.add_paragraph('The system chooses actions as: plan -> retrieve each sub-question -> dedup contexts -> synthesize answer. The graph version expands this into plan_node, retrieve_node, generate_node, reformulate_node. If planning fails, the code defaults to the original question and still continues.')
doc.add_paragraph('Bad plans are prevented mostly by constrained heuristics, not a validator. This is lightweight and deterministic, but it is not robust to nuanced decomposition. The project uses faithfulness gating as a downstream validation mechanism.')
doc.add_paragraph('Interview answers:')
add_bullets([
    'Why do you need a planner if the LLM can directly answer? Because the planner reduces the risk of retrieving irrelevant evidence and helps isolate multiple facts into more targeted retrieval steps. The system does not rely on the LLM to invent the strategy; it makes a narrow, deterministic decomposition first.',
    'Why not just call the retriever every time? A single broad query can produce mixed or generic retrieval. Planning makes multi-hop evidence gathering explicit and improves grounding quality.',
    'How do you prevent loops? The loop is bounded by max_iterations and the faithfulness threshold. If the answer is good enough or iteration limit is reached, the loop exits.',
    'How do you validate a generated plan? Currently, validation is light: the planner is heuristically constrained and the answer is later scored by grounding_score. A stronger system would add a plan validator or use LLM-as-judge scoring.',
    'Would you use an LLM planner in production? Possibly, but only with guardrails and plan validation because autonomous planning can become unstable. The current project uses a deterministic heuristic by default because it is simpler, cheaper, and easier to test.'
])

# Section 7
section('7. RETRIEVAL')
doc.add_paragraph('The retrieval subsystem is the evidence engine of the project. It combines lexical and vector retrieval, then fuses them.')
doc.add_paragraph('Document loading: load_directory(path) loads .md and .txt files and wraps each file into a Document with doc_id and text. Chunking: fixed_size_chunks creates overlapping windows with default size 120 and overlap 24; markdown_section_chunks creates sections based on headings. Metadata: each Chunk includes id, text, and metadata like doc_id, section, start, strategy. Embeddings: HashingEmbedder uses feature hashing to produce deterministic sparse vectors; OpenAIEmbedder is a real dense option. Vector search: VectorStore.search computes cosine similarity against each stored chunk vector. Ranking: BM25.search uses IDF-based scoring with tokenization and stemming. The hybrid path in HybridRetriever.retrieve uses reciprocal rank fusion (RRF) with 1/(k+rank). A rerank step then boosts matches with lexical overlap and exact phrase matches.')
doc.add_paragraph('Why each design decision exists: chunk overlap reduces boundary cuts, section-based chunking helps document structure, BM25 helps lexical precision, vector search helps semantic matching, RRF balances the two, and reranking helps prioritize phrase matches and query vocabulary in the small candidate set.')
doc.add_paragraph('Retrieval problems and project handling: poor chunking -> fixed overlap plus section chunking; semantic mismatch -> hybrid retrieval with vector search; irrelevant chunks -> rerank and grounding check; missing context -> planner decomposes the question; duplicate chunks -> dedup by chunk.id in the agent; context overflow -> top_k limits and rerank; retrieval latency -> small in-memory index, vector cache. It is not a production-scale vector database; it is a compact reference implementation.')
doc.add_paragraph('Reranking: [IMPLEMENTED] in HybridRetriever._rerank, which adds overlap and exact phrase bonus. If reranking did not exist, the system would still work, but would be more sensitive to lexical mismatch. A production extension would use a cross-encoder reranker or LLM judge to reorder candidates before synthesis.')

# Section 8
section('8. GUARDRAILS')
doc.add_paragraph('Guardrails in this project are intentionally lightweight but operationally important. The file guardrails.py contains input checks and output checks. check_input rejects overly long queries and pattern-based prompt injection strings such as “ignore all previous instructions” and “reveal the system prompt” or API key. This is a fail-closed pattern and raises InputBlocked. redact_pii masks SSNs, cards, and email addresses in final output. grounding_score computes a simple overlap ratio between answer token set and context token set.')
rows = [
    ['Guardrail type', 'Implementation', 'What it protects against'],
    ['Input', 'check_input maxlength + prompt-injection regex', 'Oversized queries; obvious prompt injection attempts'],
    ['Output', 'redact_pii()', 'Leakage of emails, card numbers, and SSNs'],
    ['Grounding', 'grounding_score()', 'Answer terms that are not supported by context'],
    ['Retrieval context', 'Agent dedup and faithfulness gate', 'Low-coverage retrieval that causes unsupported answer generation'],
    ['Tool', 'Optional gateway in integrations.py', 'JWT auth, rate limits, injection inspection, and audit when installed']
]
add_table(rows)
doc.add_paragraph('Current Protection vs Production-Grade Protection:')
rows2 = [
    ['Area', 'Current project', 'Production-grade extension'],
    ['Prompt injection', 'Regex patterns and length checks', 'Semantic classifiers, policy enforcement, allowlists, red-team tests'],
    ['Tool safety', 'Optional gateway integration', 'Authorization, least privilege, execution sandboxing, approval workflows'],
    ['Hallucination', 'Grounding score + faithfulness threshold', 'LLM judge, retrieval abstention, citation verification, answer abstain policy'],
    ['Sensitive data', 'PII redaction only', 'Data classification, encryption, per-tenant isolation, key management'],
    ['Malicious documents', 'Not implemented', 'Document scanning, content policy enforcement, provenance, sanitization']
]
add_table(rows2)
doc.add_paragraph('Interview answers: “How do you prevent prompt injection?” -> Use regex-based detection at the input boundary and fail-closed. “Can a retrieved document manipulate the agent?” -> In the current code, a malicious document could influence the model if it is retrieved and then included in context; the project does not robustly defend against indirect prompt injection in retrieved content. “How do you prevent unauthorized tool calls?” -> The optional gateway integration adds auth and audit, but the default project does not enforce a hard auth policy. “How do you validate tool arguments?” -> The gateway layer defines JSON schema for query arguments; the project’s default RetrievalTool is a simple Python method with no external schema validation beyond Python typing. “How do you stop infinite loops?” -> max_iterations and faithfulness threshold bound the loop. “How do you handle malicious documents?” -> Not fully addressed in current code; needs content scanning and policy enforcement.')

# Section 9
section('9. TOOLS')
doc.add_paragraph('The project exposes retrieval as a first-class tool via RetrievalTool in agents/tools.py. The tool is tiny: it wraps a HybridRetriever and exposes a callable interface with name “search_knowledge_base” and description “Search the indexed corpus and return the most relevant chunks.” The agent invokes it for each sub-question. This is tool calling in a minimal but genuine sense: the LLM is not calling arbitrary code; it is invoking the retrieval tool with a string argument, and the output is a list of scored chunks.')
doc.add_paragraph('For each tool, the relevant properties are: purpose: fetch relevant evidence; input schema: the retrieval tool accepts a single string query; output: list[Scored] results; invocation mechanism: the Python method call self.tool(sub_question); validation: no explicit schema validation in the default implementation, though the gateway optional wrapper enforces a JSON object schema; error handling: a failed tool call would likely propagate an exception or return an empty set; permissions: absent in default code, only in optional secure gateway; interaction with the agent: the retrieved result enters the agent state and is deduplicated before generation.')
doc.add_paragraph('The generic flow is: LLM -> tool decision -> tool call -> tool results -> state update -> next decision. This makes the system more agentic because the agent is not simply passively consuming a fixed prompt; it is actively choosing a tool boundary to gather evidence before answering.')

# Section 10
section('10. GRAPH / ORCHESTRATION')
doc.add_paragraph('The graph/orchestration logic is in src/production_agentic_rag/graph.py. The optional LangGraph engine builds a StateGraph with plan, retrieve, generate, and reformulate nodes. It uses START -> plan -> retrieve -> generate, then a conditional edge: if faithfulness >= threshold or iteration >= max_iterations, route to END; otherwise route back to reformulate then plan.')
diagram = '''
START
  |
  v
plan
  |
  v
retrieve
  |
  v
generate
  |
  +--> if faithfulness >= threshold or iterations >= max_iterations => END
  |
  +--> else => reformulate -> plan
'''
p = doc.add_paragraph()
p.add_run(diagram)
p.runs[0].font.name = 'Consolas'
p.runs[0].font.size = Pt(10)
doc.add_paragraph('Nodes and transitions: plan_node calls plan(state.get(
, state[
])); retrieve_node runs the retrieval tool across all sub-questions; generate_node builds the context prompt and calls the LLM; reformulate_node concatenates context text and re-asks the question with expansion; should_continue makes the routing decision. The state is a TypedDict with question, query, sub_questions, contexts, citations, answer, faithfulness, iterations. When retrieval fails, the system might produce empty context and a low grounding score; when planner fails, the original question remains; when LLM fails, it is an exception at the provider layer; when guardrail rejects output, the pipeline is effectively blocked before final response; when no relevant documents are found, grounding_score remains low and the agent may reformulate or return an ungrounded answer depending on model behavior.')

# Section 11
section('11. LLM + EMBEDDING PROVIDERS')
doc.add_paragraph('The project abstracts provider logic behind protocols. LLMProvider in providers/llm.py requires complete(system, user) -> LLMResult. Embedder in providers/embeddings.py requires embed(text) -> dict[int, float]. Why not hard-code one LLM? Because the project is intentionally portable between offline test modes and real cloud models, and the README emphasizes provider neutrality. The default is a deterministic MockLLM, which is ideal for tests and CI. OpenAILLM reads AZURE_OPENAI_ENDPOINT to decide whether to use AzureOpenAI or OpenAI. build_llm and build_embedder register the provider selection. This is simple and effective for a reference project, but not a complete abstraction for model routing, retries, or safety policies.')
doc.add_paragraph('Model abstraction, configuration, and design trade-offs: switching providers is a matter of setting RAG_LLM_PROVIDER, RAG_EMBED_PROVIDER, and the associated model names. This reduces lock-in and allows offline testing. It also keeps the rest of the code independent of vendor-specific APIs. The cost, latency, reliability, and fallback behaviors are not fully abstracted; there is no provider-aware retry policy or multi-model routing. The code keeps it simple, which is appropriate for a readable demonstration of architecture.')

# Section 12
section('12. INGESTION')
doc.add_paragraph('Raw documents are loaded with load_file or load_directory in ingestion/loaders.py. Each file is read as UTF-8 text and wrapped as a Document(doc_id, text). Chunking is performed by CHUNKERS with fixed-size and markdown-section strategies. fixed_size_chunks splits tokenized words into overlapping windows; markdown_section_chunks splits on markdown heading boundaries. Each chunk gets a stable id like “doc_id:0”. The ingestion lifecycle therefore is raw document -> parse -> normalize by chunker -> metadata -> retrieval index.')
doc.add_paragraph('Chunking is one of the most important parts of RAG quality because it defines the atomic evidence units. Too large and retrieval loses specificity; too small and context lacks narrative coherence. This project uses two strategies deliberately: fixed-size overlap for general text and markdown section chunking for structured docs. The selection is controlled by the strategy parameter on pipeline.add_document. It is not a production-grade semantic chunker, but it is clearly designed to respect structure and retrieval quality.')

# Section 13
section('13. EVALUATION')
doc.add_paragraph('The project’s evaluation layer sits in evaluation/harness.py and evaluation/metrics.py. The dataset is a JSONL file with question, answer, and relevant_ids. run_eval(pipeline, cases, k) iterates through each case, calls pipeline.query, then computes retrieval metrics and answer quality measures. The retrieval metrics include hit_rate, MRR, recall_at_k, and nDCG_at_k. Generation metrics include faithfulness (answer token overlap with context) and answer_correctness (token-F1 against the reference answer). This is a strong, code-level evaluation harness for RAG quality in a small system.')
doc.add_paragraph('Difference between unit tests, integration tests, RAG evaluation, LLM evaluation, and end-to-end evaluation: unit tests verify functions like tokenization and BM25; integration tests verify that components compose correctly; RAG evaluation checks retrieval and answer quality on labeled data; LLM evaluation often uses judge models or human review; end-to-end evaluation in this project is pipeline.query over a dataset plus metrics.')

# Section 14
section('14. OBSERVABILITY')
doc.add_paragraph('Observability is built around the Tracer in observability.py. It creates spans with names like plan, retrieve, synthesize, and agent.run. It captures start/end time, nested span relationships, and token usage. record_usage accumulates prompt_tokens, completion_tokens, and cost_usd using a hardcoded price book. This is a very useful debugging primitive and a lightweight production-ready foundation, though it is still local-memory-based and not a distributed tracing system.')
doc.add_paragraph('What is monitored in the code: span count, token counts, cost estimates, and nested execution log. In production, relevant metrics would also include request latency, retrieval latency, LLM latency, empty retrieval rate, tool failure rate, guardrail rejection rate, loop count, and error rate. The project currently supports only the local in-memory trace summary; bigger systems would need vendor traces, structured logs, and metrics export.')

# Section 15
section('15. TESTING')
doc.add_paragraph('The tests cover the actual behavior of the project:')
add_bullets([
    'test_agent.py verifies planner decomposition, multi-hop citations, and grounded single-hop answers.',
    'test_guardrails.py verifies input blocking, PII redaction, and grounding_score behavior.',
    'test_integrations.py verifies observability export fallback and optional gateway integration.',
    'test_langgraph.py verifies the optional LangGraph engine and fallback to the built-in agent.',
    'test_pipeline.py verifies query grounding, input injection blocking, and evaluation harness quality.',
    'test_retrieval.py verifies chunk overlap, BM25 ranking, and hybrid retrieval behavior.',
    'test_text.py verifies stemming, stopword removal, and cosine similarity.'
])
doc.add_paragraph('Overall strategy: a compact offline test suite that exercises the real retrieval and generation pipeline without API keys. This is a very good practice for a reference implementation because it proves the logic and evaluation metrics work deterministically.')

# Section 16
section('16. API')
doc.add_paragraph('The API is in api/main.py and exposes a minimal FastAPI service with /health and /ask. AskRequest requires a question string; AskResponse returns answer, citations, faithfulness, iterations, and blocked. get_pipeline lazily creates a shared RAGPipeline and preloads the corpus from data/corpus. This is a straightforward service wrapper over the same pipeline logic; no auth, queueing, or streaming is implemented. The main production concern is that it is a single-process single-tenant service in its current form.')

# Section 17
section('17. DESIGN DECISIONS')
doc.add_paragraph('Why agentic RAG? Because a single retrieval step is often insufficient when a question contains multiple facts. The project explicitly adds planning and self-correction. Why planner? Because multi-hop decomposition improves retrieval quality and reduces context mismatch. Why graph orchestration? Because the same workflow can be represented graphically and conditionally routed with a clear retry loop. Why provider abstraction? Because the system must run offline and also on real models. Why guardrails? Because prompt injection and output leakage are real failure modes for LLM systems. Why evaluation harness? To prevent the team from shipping a random prompt pipeline without measurement. Why separate ingestion? Because retrieval quality depends on chunk boundaries and metadata. Why separate retrieval? Because retrieval is a distinct subsystem with its own metrics. Why observability? Because debugging LLM systems without trace context is nearly impossible. Why tools? Because retrieval as a tool creates a proper agent boundary, making the system behave more intentionally and traceably.')

# Section 18
section('18. TRADE-OFFS')
doc.add_paragraph('Complexity vs capability: the project is more complex than basic RAG but more resilient. Latency vs reasoning: each iteration adds latency and token cost; a strict max_iterations prevents runaway behavior. Cost vs quality: more retrieval and more LLM calls improve quality but cost more. Flexibility vs determinism: heuristic planner and scorer are deterministic, but less flexible than an LLM planner. Agent autonomy vs control: the current system is relatively controlled; it is agentic enough for multi-hop retrieval but not autonomous enough to suddenly execute arbitrary actions. More guardrails vs usability: strict validation reduces unsafe behavior but can reject borderline prompts. More retrieval vs context size: the project limits result count with top_k, which controls context length. More tools vs attack surface: every additional tool increases security and maintenance complexity. LLM reasoning vs deterministic logic: the project chooses a simple deterministic workflow for indexing and planning while using the LLM only for synthesis and self-correction; this is a deliberate balance.')
doc.add_paragraph('The architecture may be overkill for a tiny FAQ or a small static knowledge base. Basic RAG would be sufficient when the question distribution is narrow, retrieval quality is high, and one retrieval step is enough. The project’s value shows up when the corpus is messy, the questions are multi-hop, and the system must be observable and testable.')

# Section 19
section('19. FAILURE MODES')
rows = [
    ['Failure', 'Detection', 'Cause', 'Recovery', 'User impact'],
    ['Empty retrieval', 'No hits in BM25 or vector store', 'No relevant chunk matched', 'Grounding score low; may reformulate or return a limited answer', 'The answer may be “I do not have enough grounded context”'],
    ['Bad retrieval', 'Low grounding_score', 'Chunk boundaries or keyword mismatch', 'Reformulate and retry; risk remains', 'Answer may be partially true but weakly supported'],
    ['LLM timeout', 'Provider exception', 'Model or network failure', '[NOT IMPLEMENTED] retry/backoff logic', 'Request may fail or be delayed'],
    ['API failure', 'FastAPI or client error', 'Transport or service issue', '[RECOMMENDED EXTENSION] circuit breaker and retry', 'User sees error or timeout'],
    ['Tool failure', 'retrieval tool exception', 'Bad tool invocation or retriever failure', 'Current code has minimal handling; default is to surface exception', 'Request fails'],
    ['Invalid tool arguments', 'Schema mismatch', 'Bad call or injection', 'Optional gateway validates schema', 'Request rejected'],
    ['Planner failure', 'Fallback to original query', 'Regex decomposition misses complexity', 'Continue with original question', 'Suboptimal retrieval quality'],
    ['Guardrail rejection', 'InputBlocked or PII redaction', 'Unsafe prompt or leakage risk', 'Block or sanitize', 'User gets blocked request or redacted answer'],
    ['Context overflow', 'Large context before generation', 'Too many retrieved chunks', 'top_k and rerank limit retrieval size', 'Truncation or reduced helpfulness'],
    ['Hallucination', 'Strong answer but weakly supported text', 'No grounding or broad mismatched retrieval', 'Grounding score and abstention policy', 'Risk of unsupported answer'],
    ['Prompt injection', 'Regex detection', 'Malicious user prompt', 'InputBlocked', 'Request blocked'],
    ['Infinite loop', 'Repeated low-faithfulness answers', 'No termination condition', 'max_iterations + threshold bound it', 'Runtime limited'],
    ['Malformed documents', 'Chunker or parser issue', 'Bad markdown or encoding', 'Loaders read raw text; no sanitization', 'Possible retrieval noise']
]
add_table(rows)

# Section 20
section('20. SECURITY')
doc.add_paragraph('The project is safer than a naive LLM app but still not enterprise-grade. Current protections: prompt-injection detection via regex on input, length limit, PII redaction on outputs, optional gateway for secure retrieval tool calls, and basic trace logging. However, it does not include robust authN/authZ, multi-tenant isolation, document provenance validation, sandboxing, or advanced malicious content filtering. A retrieved document can influence the answer if it contains instructions or manipulative text, because the project does not robustly distinguish user prompt instructions from corpus instructions. That is a real risk and a common interview challenge.')
doc.add_paragraph('For enterprise production: add authenticated API access, per-user/per-tenant authorization, RBAC for tool access, sandboxed tool execution, secret management, content scanners, and more advanced prompt-injection defenses based on retrieval filtering and context provenance. The optional integrations path is the right direction, but the base code is still a reference implementation.')

# Section 21
section('21. PRODUCTIONIZATION')
doc.add_paragraph('If this had to serve 10,000 users, the changes would not be small. Current system: in-memory index, local tracer, single-process pipeline, simple tool wrappers, deterministic offline defaults, no async processing or queueing. Production evolution: persistent vector DB, async request handling, caching, rate limiting, authN/authZ, queue workers, model routing, circuit breakers, request timeouts, model fallback, secret managers, and robust CI/CD. The project is best described as a reference architecture with a production-conscious design, not a full cloud deployment.')

# Section 22
section('22. INTERVIEW QUESTIONS')
questions = [
    ('LEVEL 1 — Basic understanding', 'What is this project?', 'It is a production-aware agentic RAG system that loads a corpus, retrieves relevant chunks, plans queries, and grounds an answer with citations. It is designed to be testable and observable rather than just a one-shot proof of concept.'),
    ('LEVEL 2 — Architecture', 'How is the project organized?', 'It is organized around providers, ingestion, retrieval, agents, guardrails, observability, evaluation, and API. The top-level RAGPipeline composes them.'),
    ('LEVEL 3 — RAG', 'Why is retrieval hybrid?', 'Because lexical and semantic matching catch different failure modes; BM25 helps keyword precision while vector search helps semantic similarity.'),
    ('LEVEL 4 — Agentic AI', 'What makes this agentic?', 'It decomposes questions, retrieves iteratively, reformulates on low grounding, and uses a tool-like retrieval interface.'),
    ('LEVEL 5 — LLMs', 'Why use a mock LLM?', 'To make the system offline and testable in CI and for deterministic evaluation.'),
    ('LEVEL 6 — Retrieval', 'What is RRF?', 'Reciprocal Rank Fusion: combine rankings from BM25 and vector search without requiring a single global score.'),
    ('LEVEL 7 — Guardrails', 'What is a guardrail here?', 'Length checks, injection blocking, PII redaction, and grounding scoring.'),
    ('LEVEL 8 — Evaluation', 'How do you know it works?', 'By running a labeled dataset through the evaluation harness and checking retrieval and faithfulness metrics.'),
    ('LEVEL 9 — Production engineering', 'What would you change for scale?', 'Persistent vector DB, auth, cache, async queueing, safer model routing, and stricter observability.'),
    ('LEVEL 10 — HARD CROSS-QUESTIONS', 'Why is the architecture not “just RAG with an extra LLM”?', 'Because it adds explicit planning, multi-hop retrieval, tool interfaces, statefulness, retries, and measurable grounding gates. That is agentic workflow design, not just a bigger prompt.')
]
for q in questions:
    doc.add_paragraph(f'{q[0]}')
    doc.add_paragraph(f'Q: {q[1]}')
    doc.add_paragraph(f'A: {q[2]}')

# Section 23
section('23. INTERVIEWER ATTACK MODE')
attack_chains = [
    ('Interviewer: “Why is this agentic RAG? Isn’t it just RAG with an extra LLM?”', 'Candidate: “It is more than just a bigger prompt. The project explicitly decomposes the question, retrieves sub-questions separately, deduplicates citations, and can retry with reformulated context when grounding is weak. That is a control loop over retrieval and generation, not a single retrieval call.”'),
    ('Interviewer: “Then why do you need the planner?”', 'Candidate: “Because a single broad query often produces noisy or mixed evidence. Planning helps isolate multiple facts and improves focus, especially in multi-hop questions.”'),
    ('Interviewer: “Does the planner use an LLM?”', 'Candidate: “In the current code, no. It is a deterministic heuristic. The interface is easy to swap to an LLM-based planner later.”'),
    ('Interviewer: “What stops it from calling dangerous tools?”', 'Candidate: “The default implementation only exposes a retrieval tool and does not have arbitrary tool execution. The optional gateway integration adds JWT auth, schema validation, and audit controls.”'),
    ('Interviewer: “What happens if retrieval returns garbage?”', 'Candidate: “The answer’s grounding score is low, so the agent can reformulate and retry. If the evidence remains weak, the project still exposes the final answer with a faithfulness estimate rather than silently pretending it is trustworthy.”'),
    ('Interviewer: “How do you know your RAG is actually working?”', 'Candidate: “I use a labeled evaluation harness and compute retrieval hit rate, MRR, recall@k, nDCG, faithfulness, and answer correctness on the dataset.”'),
    ('Interviewer: “Why should I believe the evaluation?”', 'Candidate: “Because the repo runs offline with a deterministic mock LLM and a clear dataset, so the metrics are reproducible. The code is designed to show real logic rather than a demonstration-only narrative.”'),
    ('Interviewer: “Does using LangGraph automatically make it agentic?”', 'Candidate: “No. LangGraph is only the orchestration framework. The agentic behavior comes from the planning, retrieval, and grounding loop, not just the graph library.”'),
    ('Interviewer: “Why not put all documents in the context?”', 'Candidate: “Because context windows are bounded and retrieval is needed to keep prompts precise and efficient. Also, the system is designed to select only relevant evidence.”'),
    ('Interviewer: “Why do we need embeddings if BM25 is already there?”', 'Candidate: “BM25 is lexical and keyword-oriented; embeddings help match semantic paraphrases and synonyms. The hybrid design combines both.”')
]
for chain in attack_chains:
    doc.add_paragraph(chain[0])
    doc.add_paragraph(chain[1])

# Section 24
section('24. TRICK QUESTIONS')
trick_questions = [
    'Why isn’t every RAG system agentic? The answer is that agentic behavior requires explicit control flow, planning or tool use, not just any retrieval system.',
    'Does using LangGraph automatically make something agentic? No. It is only a workflow engine; the system still needs meaningful states, decisions, and tools.',
    'Does tool calling automatically make something autonomous? No. Tool use only becomes more agentic when the system has a decision loop and safety boundaries.',
    'Why not simply use a larger context window? Because that increases cost and does not solve retrieval precision or evidence selection well.',
    'Why not put all documents into the prompt? Because the system would be more expensive, slower, and less modular.',
    'Why do we need embeddings? To handle semantic similarity beyond exact keyword overlap.',
    'Why can good embeddings still produce bad retrieval? Because chunking, metadata, query mismatch, and reranking still matter.',
    'Why can high recall still produce bad answers? Because the model may still answer with weakly supported or irrelevant evidence.',
    'Why are guardrails not enough for security? Because they are often boundary checks, not full defense-in-depth.',
    'Why isn’t evaluation accuracy alone sufficient? Because latency, cost, safety, and fail-open behavior also matter.'
]
add_bullets(trick_questions)

# Section 25
section('25. KEY CONCEPTS I MUST KNOW')
terms = [
    'Python concepts: dataclasses, protocols, type hints, environment-variable configuration, path loading, and module boundaries.',
    'Async concepts: not heavily used in this repository; it remains primarily synchronous. Async would be a production extension.',
    'Pydantic: used in the FastAPI API layer through AskRequest and AskResponse models.',
    'APIs: FastAPI endpoints /health and /ask.',
    'LLMs: mock + OpenAI/Azure adapters behind a common interface.',
    'Embeddings: hashing embedder and OpenAI embedder, each implementing a shared embed method.',
    'Vector search: cosine similarity over chunk vectors with cache.',
    'Chunking: overlap-aware segmentation of raw docs for better evidence retrieval.',
    'RAG: retrieval of grounded context followed by generation.',
    'Agentic RAG: query planning, retrieval loops, tool use, and reformulation.',
    'Planning: heuristic decomposition in planner.py.',
    'Tool calling: retrieval is a tool that the agent invokes to fetch evidence.',
    'Graphs: the optional LangGraph flow models state transitions.',
    'State: stores question, contexts, citations, answer, faithfulness, iterations.',
    'Guardrails: input rejection, PII redaction, grounding score.',
    'Prompt injection: malicious instructions in the user query or retrieved content.',
    'Evaluation: labeled QA tests next to metric functions.',
    'Observability: span-based tracing and token/cost accounting.',
    'Testing: offline deterministic CI suite with real pipeline behavior.',
    'Docker: containerization support is present in the file system but not deeply described here.',
    'Provider abstraction: the project is designed to swap providers without changing business logic.'
]
add_bullets(terms)

# Section 26
section('26. WHAT I SHOULD SAY ABOUT THE PROJECT')
doc.add_paragraph('“I studied and worked with a production-aware agentic RAG system built around a central RAGPipeline. The project loads a corpus, chunks and indexes it, then uses hybrid retrieval combining BM25 and vector similarity with reciprocal rank fusion. The user query is decomposed by a planner, each sub-question is routed through a retrieval tool, and the system synthesizes a grounded answer while tracking faithfulness and citations. It includes guardrails for prompt-injection and PII, plus a built-in evaluation harness and tracer-based observability. The core runs offline with a mock LLM and hashing embeddings, which makes it easy to test and debug in CI before connecting to OpenAI or Azure. The design is intentionally modular: providers, ingestion, retrieval, agent logic, evaluation, and API concerns are separated so each can be swapped or extended without changing the core flow.”')

# Section 27
section('27. PROJECT WALKTHROUGH SCRIPT')
doc.add_paragraph('“Let me start with the repository structure. The project is rooted around src/production_agentic_rag, which contains providers, ingestion, retrieval, agents, evaluation, and core orchestration files. I’ll begin with the ingestion layer and show how load_directory and chunking work, then move into the retrieval layer to explain BM25 + vector search + RRF, then the agent layer to describe planning and self-correction, then the guardrails, and finally the evaluation and observability layer.”')
doc.add_paragraph('Suggested file order: config.py, pipeline.py, guardrails.py, agents/planner.py, agents/rag_agent.py, agents/tools.py, retrieval/hybrid.py, retrieval/bm25.py, retrieval/vectorstore.py, ingestion/chunking.py, ingestion/loaders.py, providers/llm.py, providers/embeddings.py, graph.py, evaluation/harness.py, evaluation/metrics.py, observability.py, api/main.py, tests/test_pipeline.py.')

# Section 28
section('28. 1-DAY INTERVIEW PREPARATION')
doc.add_paragraph('Hour 1: architecture and execution flow; Hour 2: retrieval and chunking; Hour 3: planning, agent loop, and graph; Hour 4: guardrails and security; Hour 5: evaluation and observability; Hour 6: mock interview and red-team questions.')
doc.add_paragraph('MUST KNOW: RAGPipeline.query, AgenticRAG.run, HybridRetriever.retrieve, grounding_score, planner.plan, Settings, and the difference between built-in agent and LangGraph engine.')
doc.add_paragraph('SHOULD KNOW: BM25, RRF, chunking strategies, optional integrations, API schema, evaluation metrics, and failure/recovery behavior.')
doc.add_paragraph('NICE TO KNOW: production changes for scale, gateway integration, and multi-tenant auth design.')

# Section 29
section('29. FINAL CHEAT SHEET')
doc.add_paragraph('Architecture diagram: User -> API/CLI -> RAGPipeline -> guardrails -> AgenticRAG -> retrieval -> LLM -> grounding -> answer -> trace/eval.')
doc.add_paragraph('End-to-end flow: input check -> plan -> per-hop retrieval -> deduplicate -> synthesize -> faithfulness gate -> reformulate or return -> redact PII -> return QueryResult.')
doc.add_paragraph('Important files: pipeline.py, graph.py, agents/planner.py, agents/rag_agent.py, agents/tools.py, retrieval/hybrid.py, guardrails.py, observability.py, evaluation/harness.py, api/main.py.')
doc.add_paragraph('Key terminology: hybrid retrieval, reciprocal rank fusion, grounding, faithfulness threshold, max_iterations, prompt injection, PII redaction, citations, eval harness, tracer spans.')
doc.add_paragraph('10 most important design decisions: offline-first default, provider abstraction, hybrid retrieval, agentic loop, faithfulness gate, guardrails, evaluation harness, tracer-based observability, modular code layout, optional LangGraph engine.')
doc.add_paragraph('20 likely interview questions: 1) Why is this agentic? 2) Why use hybrid retrieval? 3) Why not just use a larger prompt? 4) How does the planner work? 5) What are the guardrails? 6) What is faithfulness? 7) What happens on low grounding? 8) How is the graph structured? 9) Why use a mock LLM? 10) What is the role of PII redaction? 11) Is LangGraph required? 12) What does the API look like? 13) How do you evaluate retrieval? 14) How do you test it? 15) What are the failure modes? 16) Why not just use one retrieval step? 17) What is RRF? 18) How do you do prompt injection checks? 19) What is not implemented? 20) What would you change for production?')
doc.add_paragraph('Common failure modes: empty retrieval, bad retrieval, low faithfulness, injection attempts, tool errors, context overflow, and model/provider failures.')
doc.add_paragraph('Security points: guardrails, gateway integration, least-privilege tool boundaries, sensitive data redaction, prompt-injection blocking.')
doc.add_paragraph('Production points: persistent vector DB, auth, queueing, caching, retries, rate limits, observability exports, and deployment automation.')
doc.add_paragraph('30-second explanation: “This is a provider-agnostic, agentic RAG system with hybrid retrieval, multi-hop planning, grounding checks, guardrails, and evaluation.”')
doc.add_paragraph('2-minute explanation: “It loads the corpus, indexes chunks, plans sub-queries, retrieves evidence, synthesizes with an LLM, checks grounding, and retries when necessary.”')
doc.add_paragraph('5-minute explanation: “The pipeline is built around RAGPipeline. It uses BM25 + vector retrieval with RRF, the planner decomposes queries, the agent invokes the retrieval tool, faithfulness gates self-correction, and the project measures quality with an evaluation harness.”')

out = 'INTERVIEW_MASTER_GUIDE_production_agentic_rag.docx'
doc.save(out)
print(f'Wrote {out}')